# 1) Imports


In [1]:
import json
import pickle
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# 2) Feature Engineering Helpers


In [2]:
DATE_COLS = {
    "transaction_date",
    "membership_expire_date",
    "registration_init_time",
    "expiration_date",
    "date",
}


def load_first_existing(data_dir: Path, names: Iterable[str]) -> pd.DataFrame:
    for name in names:
        path = data_dir / name
        if path.exists():
            return pd.read_csv(path)
    raise FileNotFoundError(f"None of these files were found in {data_dir}: {list(names)}")


def parse_dates(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c in out.columns:
        if c in DATE_COLS:
            out[c] = pd.to_datetime(out[c], format="%Y%m%d", errors="coerce")
    return out


def build_transaction_features(transactions: pd.DataFrame) -> pd.DataFrame:
    tx = parse_dates(transactions)
    tx = tx.sort_values(["msno", "transaction_date"]).copy()

    latest = tx.groupby("msno", as_index=False).tail(1)

    aggs = tx.groupby("msno").agg(
        tx_count=("msno", "size"),
        avg_plan_days=("payment_plan_days", "mean"),
        avg_list_price=("plan_list_price", "mean"),
        avg_paid=("actual_amount_paid", "mean"),
        auto_renew_rate=("is_auto_renew", "mean"),
        cancel_rate=("is_cancel", "mean"),
        last_tx_date=("transaction_date", "max"),
        last_membership_expire=("membership_expire_date", "max"),
    )

    aggs["days_between_tx_and_expire"] = (
        aggs["last_membership_expire"] - aggs["last_tx_date"]
    ).dt.days

    latest = latest[[
        "msno",
        "payment_method_id",
        "payment_plan_days",
        "plan_list_price",
        "actual_amount_paid",
        "is_auto_renew",
        "is_cancel",
    ]].rename(columns={
        "payment_method_id": "latest_payment_method_id",
        "payment_plan_days": "latest_payment_plan_days",
        "plan_list_price": "latest_plan_list_price",
        "actual_amount_paid": "latest_amount_paid",
        "is_auto_renew": "latest_is_auto_renew",
        "is_cancel": "latest_is_cancel",
    })

    out = aggs.reset_index().merge(latest, on="msno", how="left")
    out["avg_discount"] = out["avg_list_price"] - out["avg_paid"]
    out["latest_discount"] = out["latest_plan_list_price"] - out["latest_amount_paid"]
    return out


def build_log_features(user_logs: pd.DataFrame) -> pd.DataFrame:
    logs = parse_dates(user_logs)
    logs = logs.sort_values(["msno", "date"]).copy()

    logs["engagement_total_plays"] = (
        logs[["num_25", "num_50", "num_75", "num_985", "num_100"]].sum(axis=1)
    )
    logs["completion_ratio"] = logs["num_100"] / (logs["engagement_total_plays"] + 1e-6)

    agg = logs.groupby("msno").agg(
        log_days=("date", "nunique"),
        avg_plays=("engagement_total_plays", "mean"),
        avg_unq=("num_unq", "mean"),
        avg_total_secs=("total_secs", "mean"),
        avg_completion_ratio=("completion_ratio", "mean"),
        last_log_date=("date", "max"),
    )

    recent_window = logs[logs["date"] >= (logs["date"].max() - pd.Timedelta(days=30))]
    recent = recent_window.groupby("msno").agg(
        recent_30d_plays=("engagement_total_plays", "sum"),
        recent_30d_secs=("total_secs", "sum"),
        recent_30d_days=("date", "nunique"),
    )

    return agg.join(recent, how="left").reset_index()


def build_member_features(members: pd.DataFrame) -> pd.DataFrame:
    mem = parse_dates(members)
    out = mem.copy()

    if "bd" in out:
        out["bd"] = out["bd"].where(out["bd"].between(10, 90), np.nan)

    if "registration_init_time" in out:
        ref_date = out["registration_init_time"].max()
        out["days_since_registration"] = (ref_date - out["registration_init_time"]).dt.days

    return out


# 3) Build Training Table


In [3]:
def create_training_table(data_dir: Path) -> pd.DataFrame:
    train = load_first_existing(data_dir, ["train_v2.csv", "train.csv"])
    transactions = load_first_existing(data_dir, ["transactions_v2.csv", "transactions.csv"])
    user_logs = load_first_existing(data_dir, ["user_logs_v2.csv", "user_logs.csv"])
    members = load_first_existing(data_dir, ["members_v3.csv", "members.csv"])

    tx_feat = build_transaction_features(transactions)
    log_feat = build_log_features(user_logs)
    mem_feat = build_member_features(members)

    df = train.merge(tx_feat, on="msno", how="left")
    df = df.merge(log_feat, on="msno", how="left")
    df = df.merge(mem_feat, on="msno", how="left")
    return df


# 4) Train Explainable Model


In [4]:
def build_model(df: pd.DataFrame):
    y = df["is_churn"].astype(int)
    msno = df["msno"]
    X = df.drop(columns=["is_churn", "msno"])

    datetime_cols = [c for c in X.columns if np.issubdtype(X[c].dtype, np.datetime64)]
    X = X.drop(columns=datetime_cols)

    cat_cols = [c for c in X.columns if X[c].dtype == "object"]
    num_cols = [c for c in X.columns if c not in cat_cols]

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]), num_cols),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]), cat_cols),
        ],
        remainder="drop",
    )

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=600, class_weight="balanced")),
    ])

    X_train, X_test, y_train, y_test, msno_train, msno_test = train_test_split(
        X, y, msno, test_size=0.2, random_state=42, stratify=y
    )

    pipeline.fit(X_train, y_train)
    test_pred = pipeline.predict_proba(X_test)[:, 1]

    metrics = {
        "roc_auc": float(roc_auc_score(y_test, test_pred)),
        "average_precision": float(average_precision_score(y_test, test_pred)),
        "n_train": int(X_train.shape[0]),
        "n_test": int(X_test.shape[0]),
    }
    return pipeline, metrics, X_train, X_test, y_train, y_test, msno_train, msno_test



# 5) Explain Why a Customer Might Leave


In [5]:
def explain_customer_risk(pipeline: Pipeline, feature_table: pd.DataFrame, top_k: int = 5) -> pd.DataFrame:
    raw = feature_table.copy()
    raw_no_dates = raw.drop(columns=[c for c in raw.columns if np.issubdtype(raw[c].dtype, np.datetime64)])

    preprocessor = pipeline.named_steps["preprocessor"]
    model = pipeline.named_steps["model"]

    transformed = preprocessor.transform(raw_no_dates)
    feature_names = preprocessor.get_feature_names_out()
    coef = model.coef_.ravel()

    contributions = transformed.multiply(coef) if hasattr(transformed, "multiply") else transformed * coef
    if hasattr(contributions, "toarray"):
        contributions = contributions.toarray()

    probs = pipeline.predict_proba(raw_no_dates)[:, 1]

    top_reason_text = []
    for i in range(contributions.shape[0]):
        row = contributions[i]
        idx = np.argsort(row)[-top_k:][::-1]
        reasons = [f"{feature_names[j]} ({row[j]:.3f})" for j in idx if row[j] > 0]
        top_reason_text.append("; ".join(reasons) if reasons else "No strong positive churn signals")

    return pd.DataFrame({
        "churn_probability": probs,
        "top_reasons": top_reason_text,
    })


# 6) Run End-to-End Training

Set your dataset path and execute this cell.


In [6]:
data_dir = Path('content')
output_dir = Path('content')
output_dir.mkdir(parents=True, exist_ok=True)

df = create_training_table(data_dir)
pipeline, metrics, X_train, X_test, y_train, y_test, msno_train, msno_test = build_model(df)

# Скоринг и объяснения делаем на тестовой выборке
explanations = explain_customer_risk(pipeline, X_test)
scored = pd.concat([
    msno_test.reset_index(drop=True).rename('msno'),
    y_test.reset_index(drop=True).rename('is_churn'),
    explanations.reset_index(drop=True),
], axis=1)
scored = scored.sort_values('churn_probability', ascending=False)

with open(output_dir / 'churn_model.pkl', 'wb') as f:
    pickle.dump(pipeline, f)
scored.head(100).to_csv(output_dir / 'top_churn_risk_with_reasons.csv', index=False)
with open(output_dir / 'validation_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

metrics



c:\Users\tomte\anaconda3\Lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['bd']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\tomte\anaconda3\Lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['bd']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\tomte\anaconda3\Lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['bd']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\tomte\anaconda3\Lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['bd']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


{'roc_auc': 0.9092773330021365,
 'average_precision': 0.6462129878395568,
 'n_train': 776768,
 'n_val': 194192}

In [7]:
scored

,msno,is_churn,churn_probability,top_reasons
610033,QFdoW0TxX6XLe5lRxBzi8OtsOLUID0JibVVWActBQMM=,1,1.000000e+00,num__avg_plan_days (78.030); num__avg_list_pri...
12745,/vKZkJwqzb2y8Nc3HiYlScE1X1E1EWAGLgOnhZ1CVQc=,1,1.000000e+00,num__avg_plan_days (78.030); num__avg_list_pri...
47361,fZ5YhbzejLIG570q10Pb2FZIyruRHqf12OMGmoOn0rI=,1,1.000000e+00,num__avg_plan_days (78.030); num__avg_list_pri...
584468,Ahjw69uPPi3wwBLNk5YzMuiiz6OMeMczFw3azqRk91E=,1,1.000000e+00,num__avg_plan_days (78.030); num__avg_list_pri...
584467,2lNDgdnv/sI99iltH6yvY7vUaUqVRznYyBDw82wHKfI=,1,1.000000e+00,num__avg_plan_days (33.469); num__avg_list_pri...
...,...,...,...,...
19476,36L0w+CaRZkm2/PPmWG4d6zBs1ppcblP4ko334F1SOk=,1,6.287731e-07,num__auto_renew_rate (4.139); num__latest_paym...
612718,uYDvhd2eekMIai+7M0GvGnemsUavV5BUhA5af/yVwrs=,1,4.346675e-07,num__auto_renew_rate (4.139); num__latest_paym...
64827,nSKjEvBhfVDdsDrTD76tnT4tfnDLVfqczBgew+Rh2h8=,0,2.905492e-07,num__auto_renew_rate (4.139); num__latest_paym...
292924,NkX9tnRFX5aX1PETmQ1BoPKHnDlJ6Zj/vutj+DIwbyg=,0,2.260444e-07,num__auto_renew_rate (4.139); num__avg_plays (...


# 7) LightGBM + SHAP Explainability (new section)
Добавляем второй пайплайн (LightGBM) на тех же признаках и том же сплите, затем сравниваем с Logistic Regression и интерпретируем модель через SHAP.

In [ ]:
# Импорты для нового пайплайна и интерпретации
import matplotlib.pyplot as plt
import shap
from lightgbm import LGBMClassifier
from sklearn.metrics import log_loss, roc_auc_score


In [ ]:
# Используем тот же train/validation split, который уже создан выше (X_train, X_test, y_train, y_test).
# Отдельно обучим LightGBM-пайплайн с аккуратной обработкой DataFrame-признаков.
cat_cols_lgbm = [c for c in X_train.columns if X_train[c].dtype == 'object']
num_cols_lgbm = [c for c in X_train.columns if c not in cat_cols_lgbm]

lgbm_preprocessor = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), num_cols_lgbm),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore')),
        ]), cat_cols_lgbm),
    ],
    remainder='drop',
)

lgbm_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
    n_jobs=-1,
)

lgbm_pipeline = Pipeline([
    ('preprocessor', lgbm_preprocessor),
    ('model', lgbm_model),
])

lgbm_pipeline.fit(X_train, y_train)
lgbm_val_pred = lgbm_pipeline.predict_proba(X_test)[:, 1]

# Для корректного сравнения считаем метрики для Logistic Regression на том же validation-наборе
lr_val_pred = pipeline.predict_proba(X_test)[:, 1]

comparison_df = pd.DataFrame([
    {
        'model': 'Logistic Regression',
        'log_loss': log_loss(y_test, lr_val_pred),
        'roc_auc': roc_auc_score(y_test, lr_val_pred),
    },
    {
        'model': 'LightGBM',
        'log_loss': log_loss(y_test, lgbm_val_pred),
        'roc_auc': roc_auc_score(y_test, lgbm_val_pred),
    },
]).sort_values('roc_auc', ascending=False)

comparison_df


In [ ]:
# Важность признаков LightGBM (model-native feature importance)
lgbm_pre = lgbm_pipeline.named_steps['preprocessor']
lgbm_clf = lgbm_pipeline.named_steps['model']
feature_names_lgbm = lgbm_pre.get_feature_names_out()
feature_importance_df = pd.DataFrame({
    'feature': feature_names_lgbm,
    'importance': lgbm_clf.feature_importances_,
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 7))
top_n = 20
plot_df = feature_importance_df.head(top_n).iloc[::-1]
plt.barh(plot_df['feature'], plot_df['importance'])
plt.title(f'LightGBM Feature Importance (Top {top_n})')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

feature_importance_df.head(10)


In [ ]:
# SHAP для LightGBM: глобальная и локальная интерпретация
# Сначала трансформируем данные тем же препроцессором, чтобы сохранить соответствие признаков.
X_train_trans = lgbm_pre.transform(X_train)
X_test_trans = lgbm_pre.transform(X_test)

# Для SHAP удобнее dense-представление, если пришла sparse-матрица
X_train_trans_dense = X_train_trans.toarray() if hasattr(X_train_trans, 'toarray') else X_train_trans
X_test_trans_dense = X_test_trans.toarray() if hasattr(X_test_trans, 'toarray') else X_test_trans

explainer = shap.TreeExplainer(lgbm_clf)
shap_values_raw = explainer.shap_values(X_test_trans_dense)

# В зависимости от версии SHAP для бинарной задачи может вернуться список из 2 массивов
if isinstance(shap_values_raw, list):
    shap_values = shap_values_raw[1]
else:
    shap_values = shap_values_raw

X_test_shap_df = pd.DataFrame(X_test_trans_dense, columns=feature_names_lgbm)

# 1) SHAP summary plot (с fallback при несовместимости версий)
try:
    shap.summary_plot(shap_values, X_test_shap_df, show=True)
except Exception as e:
    print('SHAP summary plot недоступен в текущей версии окружения. Используем fallback-ранжирование.')
    print('Причина:', e)

# 2) Объяснение одного объекта валидации
example_idx = 0
example_pred = float(lgbm_val_pred[example_idx])
print(f'Validation example index: {example_idx}')
print(f'Predicted churn probability (LightGBM): {example_pred:.4f}')

try:
    # Для современных версий SHAP
    shap.plots.waterfall(shap.Explanation(
        values=shap_values[example_idx],
        base_values=explainer.expected_value[1] if isinstance(explainer.expected_value, (list, np.ndarray)) else explainer.expected_value,
        data=X_test_shap_df.iloc[example_idx],
        feature_names=feature_names_lgbm,
    ))
except Exception as e:
    print('SHAP waterfall plot недоступен в текущей версии окружения. Покажем top-вклады текстом.')
    print('Причина:', e)
    local_contrib = pd.DataFrame({
        'feature': feature_names_lgbm,
        'shap_value': shap_values[example_idx],
        'abs_shap_value': np.abs(shap_values[example_idx]),
    }).sort_values('abs_shap_value', ascending=False)
    display(local_contrib.head(10))

# 3) Top churn drivers по mean(|SHAP|)
mean_abs_shap = np.abs(shap_values).mean(axis=0)
top_churn_drivers = pd.DataFrame({
    'feature': feature_names_lgbm,
    'mean_abs_shap': mean_abs_shap,
}).sort_values('mean_abs_shap', ascending=False)

print('Top-10 churn drivers by mean absolute SHAP (model-driven importance, NOT causal proof):')
display(top_churn_drivers.head(10))
